# 2.1 缩放点积注意力 / Scaled dot-product attention

位置之间交换信息的唯一途径。实现它，并从方差推出为什么要除以 √d_k。

The only way positions exchange information. Implement it and derive from the variance why it divides by √d_k.

讲解与推导见网站对应模块。每个练习先写代码，再运行紧随其后的检查单元。

The explanation and derivations are on the website. For each exercise, write the code, then run the check cell right after it.

## 1. 实现缩放点积注意力 / Implement scaled dot-product attention

按公式实现注意力，返回输出和注意力权重，并支持布尔掩码。测试会与 `F.scaled_dot_product_attention` 对比，包括带掩码的情形。

Implement attention from the formula, return both the output and the attention weights, and support a boolean mask. The tests compare with `F.scaled_dot_product_attention`, including the masked case.

In [ ]:
import math
import torch


def scaled_dot_product_attention(q, k, v, mask=None):
    """Attention(Q, K, V) = softmax(Q K^T / sqrt(d_k)) V

    q: (..., T_q, d_k)   k: (..., T_k, d_k)   v: (..., T_k, d_v)
    mask: optional bool tensor broadcastable to (..., T_q, T_k).
          True means "this query may look at this key".
    Returns (output, weights) with shapes (..., T_q, d_v) and (..., T_q, T_k).
    Do not call F.scaled_dot_product_attention.
    """
    # TODO 1: scores = q @ k^T, divided by sqrt(d_k)
    # TODO 2: where mask is False, set the score to -inf
    # TODO 3: softmax over the keys (the last dimension)
    # TODO 4: output = weights @ v
    raise NotImplementedError


def self_attention(x, w_q, w_k, w_v):
    """Single-head self-attention: queries, keys and values all come from x.

    x: (T, d)   w_q, w_k: (d, d_k)   w_v: (d, d_v)
    """
    # TODO 5: project x three times, then call scaled_dot_product_attention
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch
import torch.nn.functional as F

torch.manual_seed(0)
B, Tq, Tk, dk, dv = 2, 5, 7, 16, 12
q = torch.randn(B, Tq, dk, dtype=torch.float64)
k = torch.randn(B, Tk, dk, dtype=torch.float64)
v = torch.randn(B, Tk, dv, dtype=torch.float64)

out, w = scaled_dot_product_attention(q, k, v)
assert out.shape == (B, Tq, dv) and w.shape == (B, Tq, Tk), "check the shapes of output and weights"
assert torch.allclose(w.sum(-1), torch.ones(B, Tq, dtype=torch.float64)), "each query's weights must sum to 1"
assert torch.allclose(out, F.scaled_dot_product_attention(q, k, v), atol=1e-10), "differs from the PyTorch implementation"

# with a mask: True = may attend
mask = torch.rand(B, Tq, Tk) > 0.4
mask[..., 0] = True                                   # every query can see at least one key
out_m, w_m = scaled_dot_product_attention(q, k, v, mask)
assert torch.allclose(out_m, F.scaled_dot_product_attention(q, k, v, attn_mask=mask), atol=1e-10), "masked result differs from PyTorch"
assert (w_m[~mask] == 0).all(), "masked keys must get exactly zero weight"

# the scaling really is sqrt(d_k): identical q and k rows give a score of |q|^2 / sqrt(d_k)
one = torch.ones(1, 4, dtype=torch.float64)
_, w1 = scaled_dot_product_attention(one, torch.cat([one, 0 * one]), torch.zeros(2, 1, dtype=torch.float64))
assert torch.allclose(w1, torch.softmax(torch.tensor([[2.0, 0.0]], dtype=torch.float64), -1)), "divide by sqrt(d_k), not d_k"

# self-attention on a single sequence
x = torch.randn(6, 8, dtype=torch.float64)
wq, wk, wv = (torch.randn(8, 4, dtype=torch.float64) for _ in range(3))
o, a = self_attention(x, wq, wk, wv)
assert o.shape == (6, 4) and a.shape == (6, 6)
assert torch.allclose(o, F.scaled_dot_product_attention(x @ wq, x @ wk, x @ wv), atol=1e-10)
print("m21_attention: all tests passed")

<details><summary><b>提示 1 / Hint 1</b></summary>

转置最后两个维度用 `k.transpose(-2, -1)`，这样函数对带批量维度和不带批量维度的输入都适用。

Transpose the last two dimensions with `k.transpose(-2, -1)` so the function works with and without batch dimensions.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

掩码用 `scores.masked_fill(~mask, float('-inf'))`，在 softmax 之前执行。

Apply the mask with `scores.masked_fill(~mask, float('-inf'))` before the softmax.

</details>

<details><summary><b>提示 3 / Hint 3</b></summary>

$d_k$ 取 `q.shape[-1]`，除的是它的平方根。

$d_k$ is `q.shape[-1]`, and the division is by its square root.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def scaled_dot_product_attention(q, k, v, mask=None):
    d_k = q.shape[-1]
    scores = q @ k.transpose(-2, -1) / math.sqrt(d_k)        # (..., T_q, T_k): one score per (query, key) pair
    if mask is not None:
        scores = scores.masked_fill(~mask, float("-inf"))    # exp(-inf) = 0, so these keys get zero weight
    weights = torch.softmax(scores, dim=-1)                  # each query's weights sum to 1
    return weights @ v, weights                              # weighted average of the values


def self_attention(x, w_q, w_k, w_v):
    return scaled_dot_product_attention(x @ w_q, x @ w_k, x @ w_v)
```

</details>

## 2. 实验验证不缩放时 softmax 会饱和 / An experiment showing softmax saturation without scaling

用随机向量做三个测量。点积的方差是否等于 $d_k$。16 个随机 key 上的最大注意力权重随 $d_k$ 怎样变化。通过 softmax 传回的梯度有多大。测试会检查不缩放时权重随维度增大而集中到单个位置，缩放后则保持不变。

Take three measurements with random vectors. Whether the variance of the dot product equals $d_k$. How the largest attention weight over 16 random keys changes with $d_k$. How much gradient passes back through the softmax. The tests check that without scaling the weights concentrate on a single position as the dimension grows, and that with scaling they stay the same.

In [ ]:
import math
import torch


def dot_product_variance(d_k, n=20000, scaled=False, generator=None):
    """Empirical variance of q . k for random q, k with independent N(0, 1) entries.

    Draw n pairs of vectors of size d_k. If scaled, divide each dot product by sqrt(d_k).
    Returns a Python float.
    """
    # TODO 1: q, k = torch.randn(n, d_k, generator=generator) twice; dot products; .var()
    raise NotImplementedError


def mean_max_weight(d_k, T=16, n=2000, scaled=False, generator=None):
    """How peaked is softmax over T random keys?

    For n random queries (n, 1, d_k) and keys (n, T, d_k), compute the attention
    weights over the T keys and return the average of the largest weight (a float).
    1/T means perfectly flat, 1.0 means all weight on a single key.
    """
    # TODO 2
    raise NotImplementedError


def score_gradient_norm(d_k, T=16, n=500, scaled=False, generator=None):
    """How much gradient gets through the softmax?

    Build scores as above with requires_grad, take weights = softmax(scores),
    backpropagate loss = (weights * r).sum() for a fixed random r of shape (n, T),
    and return the average L2 norm of d loss / d scores over the n queries (a float).
    """
    # TODO 3: create the scores as a leaf tensor: scores = (...).detach().requires_grad_(True)
    raise NotImplementedError

In [ ]:
# 检查 / checks
import torch

g = lambda: torch.Generator().manual_seed(0)

# 1. Var(q . k) = d_k, and dividing by sqrt(d_k) brings it back to 1
for d in (16, 64, 256, 1024):
    v = dot_product_variance(d, generator=g())
    assert abs(v / d - 1) < 0.1, f"d_k={d}: variance {v:.1f} should be close to {d}"
    vs = dot_product_variance(d, scaled=True, generator=g())
    assert abs(vs - 1) < 0.1, f"d_k={d}: scaled variance {vs:.3f} should be close to 1"

# 2. without scaling, softmax collapses onto one key as d_k grows
raw = [mean_max_weight(d, generator=g()) for d in (16, 64, 256, 1024)]
assert raw == sorted(raw) and raw[-1] > 0.9, f"unscaled weights should get more peaked with d_k: {raw}"
sc = [mean_max_weight(d, scaled=True, generator=g()) for d in (16, 64, 256, 1024)]
assert max(sc) - min(sc) < 0.05 and max(sc) < 0.4, f"scaled weights should stay equally spread for every d_k: {sc}"

# 3. a saturated softmax passes little gradient
g_raw = score_gradient_norm(1024, generator=g())
g_sc = score_gradient_norm(1024, scaled=True, generator=g())
assert g_sc > 3 * g_raw, f"scaled gradient {g_sc:.3f} should be several times the unscaled {g_raw:.3f}"
print("m21_variance: all tests passed")
print("  mean max weight, unscaled:", [round(x, 3) for x in raw])
print("  mean max weight, scaled  :", [round(x, 3) for x in sc])
print(f"  gradient norm at d_k=1024: unscaled {g_raw:.3f}, scaled {g_sc:.3f}")

<details><summary><b>提示 1 / Hint 1</b></summary>

query 取形状 `(n, 1, d_k)`，key 取 `(n, T, d_k)`，相乘后对最后一维求和得到 `(n, T)` 的分数。

Use queries of shape `(n, 1, d_k)` and keys of `(n, T, d_k)`. Multiplying and summing over the last dimension gives scores of shape `(n, T)`.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

求梯度时先把分数变成叶子张量，即 `scores = scores.detach().requires_grad_(True)`，反向传播后读取 `scores.grad`。

For the gradient, make the scores a leaf tensor with `scores = scores.detach().requires_grad_(True)` and read `scores.grad` after the backward pass.

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

```python
import math
import torch


def _scores(d_k, T, n, scaled, generator):
    q = torch.randn(n, 1, d_k, generator=generator)
    k = torch.randn(n, T, d_k, generator=generator)
    s = (q * k).sum(-1)                               # (n, T)
    return s / math.sqrt(d_k) if scaled else s


def dot_product_variance(d_k, n=20000, scaled=False, generator=None):
    q = torch.randn(n, d_k, generator=generator)
    k = torch.randn(n, d_k, generator=generator)
    dots = (q * k).sum(-1)
    if scaled:
        dots = dots / math.sqrt(d_k)
    return dots.var().item()


def mean_max_weight(d_k, T=16, n=2000, scaled=False, generator=None):
    weights = torch.softmax(_scores(d_k, T, n, scaled, generator), dim=-1)
    return weights.max(dim=-1).values.mean().item()


def score_gradient_norm(d_k, T=16, n=500, scaled=False, generator=None):
    scores = _scores(d_k, T, n, scaled, generator).detach().requires_grad_(True)
    r = torch.randn(n, T, generator=generator)
    (torch.softmax(scores, dim=-1) * r).sum().backward()
    return scores.grad.norm(dim=-1).mean().item()
```

</details>

## 3. 推导除以 √d_k 的理由 / Derive the reason for dividing by √d_k

这是面试中的高频推导题。合上上面的推导，在纸上完成。
(a) 设 $q$ 和 $k$ 的各分量相互独立，均值为 0，方差为 1。证明 $\mathrm{Var}(q \cdot k) = d_k$。
(b) 如果分量的方差分别是 $\sigma_q^2$ 和 $\sigma_k^2$，结果是什么？
(c) 为什么不除以 $d_k$？
(d) 取 $d_k = 64$ 且不缩放。两个 key 的分数相差两个标准差时，它们的注意力权重之比大约是多少？

This derivation is frequent in interviews. Close the derivation above and work on paper.
(a) Let the components of $q$ and $k$ be independent with mean 0 and variance 1. Show that $\mathrm{Var}(q \cdot k) = d_k$.
(b) What is the result if the components have variances $\sigma_q^2$ and $\sigma_k^2$?
(c) Why not divide by $d_k$?
(d) Take $d_k = 64$ without scaling. When the scores of two keys differ by two standard deviations, what is the approximate ratio of their attention weights?

<details><summary><b>提示 1 / Hint 1</b></summary>

(a) 先求单项 $q_m k_m$ 的均值和方差，再利用独立变量之和的方差等于方差之和。

(a) First find the mean and variance of a single term $q_m k_m$, then use that the variance of a sum of independent variables is the sum of the variances.

</details>

<details><summary><b>提示 2 / Hint 2</b></summary>

(d) 两个 token 的权重之比是 $e^{\Delta}$，$\Delta$ 是分数之差（见 1.3）。

(d) The weight ratio of two tokens is $e^{\Delta}$, where $\Delta$ is the gap between the scores (see 1.3).

</details>

<details><summary><b>参考答案 / Reference solution</b></summary>

(a) 单项的均值为 $\mathbb{E}[q_m k_m] = \mathbb{E}[q_m]\mathbb{E}[k_m] = 0$，方差为 $\mathbb{E}[q_m^2]\mathbb{E}[k_m^2] = 1$。各项独立，所以 $\mathrm{Var}(q \cdot k) = \sum_{m=1}^{d_k} 1 = d_k$。

(b) 单项的方差变为 $\sigma_q^2 \sigma_k^2$，所以 $\mathrm{Var}(q \cdot k) = d_k\, \sigma_q^2 \sigma_k^2$。结论不变，方差仍与 $d_k$ 成正比。

(c) 除以 $d_k$ 后方差为 $d_k / d_k^2 = 1/d_k$，维度越高分数越接近。softmax 的输出接近均匀分布，注意力退化成对所有位置取平均，无法聚焦。

(d) 标准差是 $\sqrt{64} = 8$，两个标准差是 16，权重之比约为 $e^{16} \approx 8.9 \times 10^{6}$。分数较低的那个 key 实际上得不到任何权重，这就是饱和。缩放之后差距变为 2，比值约为 $e^{2} \approx 7.4$。

---

(a) A single term has mean $\mathbb{E}[q_m k_m] = \mathbb{E}[q_m]\mathbb{E}[k_m] = 0$ and variance $\mathbb{E}[q_m^2]\mathbb{E}[k_m^2] = 1$. The terms are independent, so $\mathrm{Var}(q \cdot k) = \sum_{m=1}^{d_k} 1 = d_k$.

(b) The variance of a single term becomes $\sigma_q^2 \sigma_k^2$, so $\mathrm{Var}(q \cdot k) = d_k\, \sigma_q^2 \sigma_k^2$. The conclusion is unchanged and the variance is still proportional to $d_k$.

(c) After dividing by $d_k$ the variance is $d_k / d_k^2 = 1/d_k$, so the scores move closer together as the dimension grows. The softmax output approaches uniform, and attention degrades into an average over all positions with no focus.

(d) The standard deviation is $\sqrt{64} = 8$, two standard deviations are 16, and the weight ratio is about $e^{16} \approx 8.9 \times 10^{6}$. The key with the lower score effectively gets no weight at all, which is saturation. After scaling the gap becomes 2 and the ratio about $e^{2} \approx 7.4$.

</details>